# 📁 Notebook 01 — Data Preprocessing
**Video → Frames & Audio**  
Person 1 (Lead) + Person 2 (assist)

In [ ]:
# ✅ Cell 1 — Install dependencies
!pip install opencv-python-headless moviepy librosa soundfile tqdm -q


In [ ]:
# ✅ Cell 2 — Mount Drive & Imports
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, librosa, soundfile as sf, numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ✅ Cell 3 — Config (BASE_DIR is your Google Drive project folder)
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/deepfake-project"

RAW_DIRS = {
    "fake":            os.path.join(BASE_DIR, "data/raw/fake"),
    "fake_with_audio": os.path.join(BASE_DIR, "data/raw/fake-with-audio"),
    "real":            os.path.join(BASE_DIR, "data/raw/real"),
    "real_with_audio": os.path.join(BASE_DIR, "data/raw/real-with-audio"),
}

OUT_FRAMES = os.path.join(BASE_DIR, "data/frames")
OUT_AUDIO  = os.path.join(BASE_DIR, "data/audio")
FPS_EXTRACT = 1       # frames per second to extract
MAX_FRAMES  = 300     # cap per video
AUDIO_SR    = 16000   # sample rate

for d in [OUT_FRAMES, OUT_AUDIO]:
    os.makedirs(d, exist_ok=True)

print("✅ Config ready. Folders created.")

In [ ]:
# ✅ Cell 4 — Frame Extraction Function
def extract_frames(video_path, out_dir, video_id, fps=FPS_EXTRACT):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  [WARN] Cannot open {video_path}"); return []
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, int(native_fps / fps))
    saved, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % step == 0:
            fname = os.path.join(out_dir, f"{video_id}_frame_{idx:05d}.jpg")
            cv2.imwrite(fname, frame); saved.append(fname)
            if len(saved) >= MAX_FRAMES: break
        idx += 1
    cap.release()
    return saved

print("✅ extract_frames() defined.")

In [ ]:
# ✅ Cell 5 — Run Frame Extraction on All Splits
def extract_all_frames():
    stats = {}
    for split_name, raw_dir in RAW_DIRS.items():
        if not os.path.isdir(raw_dir):
            print(f"[SKIP] {raw_dir} not found"); continue
        videos = list(Path(raw_dir).rglob("*.mp4")) + \
                 list(Path(raw_dir).rglob("*.avi")) + \
                 list(Path(raw_dir).rglob("*.mov"))
        print(f"\n=== {split_name}: {len(videos)} videos ===")
        for vp in tqdm(videos):
            vid_id  = f"{split_name}_{vp.stem}"
            out_sub = os.path.join(OUT_FRAMES, split_name)
            frames  = extract_frames(str(vp), out_sub, vid_id)
            stats.setdefault(split_name, []).append((vid_id, len(frames)))
    return stats

frame_stats = extract_all_frames()
print("\n✅ Frame extraction complete.")
for k, v in frame_stats.items():
    total = sum(x[1] for x in v)
    print(f"  {k}: {len(v)} videos → {total} frames")

In [ ]:
# ✅ Cell 6 — Audio Extraction Function
def extract_audio(video_path, out_dir, video_id, sr=AUDIO_SR):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{video_id}.wav")
    try:
        from moviepy.editor import VideoFileClip
        clip = VideoFileClip(video_path)
        if clip.audio is None:
            clip.close(); return None
        clip.audio.write_audiofile(out_path, fps=sr, logger=None)
        clip.close()
    except Exception:
        os.system(f'ffmpeg -y -i "{video_path}" -ar {sr} -ac 1 "{out_path}" -loglevel quiet')
    return out_path if os.path.exists(out_path) else None

print("✅ extract_audio() defined.")

In [ ]:
# ✅ Cell 7 — Run Audio Extraction (audio splits only)
def extract_all_audio():
    for split_name in ["fake_with_audio", "real_with_audio"]:
        raw_dir = RAW_DIRS.get(split_name)
        if not raw_dir or not os.path.isdir(raw_dir):
            print(f"[SKIP] {raw_dir}"); continue
        videos  = list(Path(raw_dir).rglob("*.mp4")) + \
                  list(Path(raw_dir).rglob("*.avi")) + \
                  list(Path(raw_dir).rglob("*.mov"))
        print(f"\n=== {split_name}: {len(videos)} videos ===")
        out_sub = os.path.join(OUT_AUDIO, split_name)
        for vp in tqdm(videos):
            vid_id = f"{split_name}_{vp.stem}"
            extract_audio(str(vp), out_sub, vid_id)
    print("\n✅ Audio extraction complete.")

extract_all_audio()

In [ ]:
# ✅ Cell 8 — Sanity Check
def count_files(directory, ext=".jpg"):
    return sum(1 for _ in Path(directory).rglob(f"*{ext}"))

print("=== Sanity Check ===")
print(f"Frames (.jpg) : {count_files(OUT_FRAMES, '.jpg')}")
print(f"Audio  (.wav) : {count_files(OUT_AUDIO,  '.wav')}")